## Approach-2: Using PDF from Roll Calls / Amendments sections from Mass legislature website
- https://malegislature.gov/Bills/194/S2538/RollCall 
- https://malegislature.gov/Bills/194/S2538/Amendments/Senate

> This code block downloads the roll-call vote PDF from MA Legislature website, convert it to image and extracts data using OCR. For each roll call session, we identify each legislator’s vote on a specific bill

In [3]:
import requests
from pdf2image import convert_from_bytes
import pytesseract
import pandas as pd

# Get PDF from URL and extract text using OCR
def extract_rollcall_text(pdf_url):
    response = requests.get(pdf_url)
    response.raise_for_status()

    # Convert PDF to images
    images = convert_from_bytes(response.content)

    # Using Tesseract to extract text from images
    text = ""
    for img in images:
        custom_config = r'--oem 3 --psm 11'     # Indicating OEM 3 for default and PSM 11 for sparse text
        text += pytesseract.image_to_string(img, config=custom_config)
    return text

# Parse the extracted text to get names and votes. 
def parse_rollcall_text(raw_text, bill_no):
    results = []
    prev=''      # used to identify what vote the name corresponds to
    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue
        if 'YEA' in line or 'NAY' in line or 'PRESENT' in line or 'ABSENT' in line:
            if 'YEA' in line: 
                prev='YEA'
            elif 'NAY' in line: 
                prev='NAY'
            elif 'PRESENT' in line: 
                prev='PRESENT'
            else: 
                prev= 'ABSENT'
        elif prev=='': 
            continue
        else:
            results.append(
                {"bill no:": bill_no, 
                "name": line.replace(prev,'').strip(), 
                "vote": prev})
    return results


pdf_url = "https://malegislature.gov/RollCall/194/SenateRollCall56.pdf"
text = extract_rollcall_text(pdf_url)
data = parse_rollcall_text(text, 'S2538')
df = pd.DataFrame(data)

# Standardizing vote values
df['vote'] = df['vote'].replace({'YEA': 'Y', 'NAY': 'N'})

# Handling suffix in names like . / -5
df['name'] = df['name'].str.replace(r'\.\s*[-–—]?\s*\d*.*$', '', regex=True).str.strip()
display(df.head())
   


,bill no:,name,vote
0,S2538,"Dooner, Kelly A",Y
1,S2538,"O'Connor, Patrick M",Y
2,S2538,"Durant, Peter J",Y
3,S2538,"Tarr, Bruce E",Y
4,S2538,"Fattman, Ryan C",Y
